In [2]:
import kagglehub
import os
import pandas as pd

# Download dataset
path = kagglehub.dataset_download("jillanisofttech/covid-19-dataset")

# path الرئيسي
data_path = os.path.join(path, "Covid-19")

print(os.listdir(data_path))

100%|██████████| 19.0M/19.0M [00:00<00:00, 133MB/s]

Extracting files...


['day_wise.csv', 'worldometer_data.csv', 'full_grouped.csv', 'usa_country_wise.csv', 'covid_19_clean_complete.csv', 'country_wise_latest.csv']


In [3]:
df_country = pd.read_csv(os.path.join(data_path, "country_wise_latest.csv"))
df_world = pd.read_csv(os.path.join(data_path, "worldometer_data.csv"))

df_clean = pd.read_csv(os.path.join(data_path, "covid_19_clean_complete.csv"))
df_full = pd.read_csv(os.path.join(data_path, "full_grouped.csv"))

df_day = pd.read_csv(os.path.join(data_path, "day_wise.csv"))
df_usa = pd.read_csv(os.path.join(data_path, "usa_country_wise.csv"))

In [4]:
import pandas as pd
import os

# =========================
# Load dataset path
# =========================
data_path = os.path.join(path, "Covid-19")

# =========================
# Function to inspect dataset
# =========================
def explore_dataset(df, name):
    print("\n" + "="*60)
    print(f"DATASET: {name}")
    print("="*60)

    # Shape
    print("Shape:", df.shape)

    # Columns
    print("\nColumns:")
    print(df.columns.tolist())

    # Data types
    print("\nData Types:")
    print(df.dtypes)

    # Missing values
    print("\nMissing Values:")
    print(df.isnull().sum())

    # Basic info
    print("\nInfo:")
    print(df.info())

    # Sample data
    print("\nHead:")
    print(df.head())

In [5]:
df_country = pd.read_csv(os.path.join(data_path, "country_wise_latest.csv"))
df_world = pd.read_csv(os.path.join(data_path, "worldometer_data.csv"))
df_clean = pd.read_csv(os.path.join(data_path, "covid_19_clean_complete.csv"))
df_full = pd.read_csv(os.path.join(data_path, "full_grouped.csv"))
df_day = pd.read_csv(os.path.join(data_path, "day_wise.csv"))
df_usa = pd.read_csv(os.path.join(data_path, "usa_country_wise.csv"))

In [6]:
explore_dataset(df_country, "country_wise_latest")
explore_dataset(df_world, "worldometer_data")
explore_dataset(df_clean, "covid_19_clean_complete")
explore_dataset(df_full, "full_grouped")
explore_dataset(df_day, "day_wise")
explore_dataset(df_usa, "usa_country_wise")


DATASET: country_wise_latest
Shape: (187, 15)

Columns:
['Country/Region', 'Confirmed', 'Deaths', 'Recovered', 'Active', 'New cases', 'New deaths', 'New recovered', 'Deaths / 100 Cases', 'Recovered / 100 Cases', 'Deaths / 100 Recovered', 'Confirmed last week', '1 week change', '1 week % increase', 'WHO Region']

Data Types:
Country/Region             object
Confirmed                   int64
Deaths                      int64
Recovered                   int64
Active                      int64
New cases                   int64
New deaths                  int64
New recovered               int64
Deaths / 100 Cases        float64
Recovered / 100 Cases     float64
Deaths / 100 Recovered    float64
Confirmed last week         int64
1 week change               int64
1 week % increase         float64
WHO Region                 object
dtype: object

Missing Values:
Country/Region            0
Confirmed                 0
Deaths                    0
Recovered                 0
Active              

In [7]:
# =========================
# 📌 Standardize column names
# =========================
def clean_columns(df):
    df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace("/", "_")
    return df

df_country = clean_columns(df_country)
df_world = clean_columns(df_world)
df_full = clean_columns(df_full)
df_clean = clean_columns(df_clean)
df_day = clean_columns(df_day)
df_usa = clean_columns(df_usa)

In [8]:
# =========================
# 📌 CLUSTERING DATASET MERGE
# =========================

# نختار الأعمدة المهمة فقط من worldometer لتقليل الضوضاء
world_subset = df_world[[
    "Country_Region",
    "Population",
    "TotalCases",
    "TotalDeaths",
    "TotalRecovered",
    "Tests_1M_pop"
]]

# توحيد اسم العمود (مهم جداً)
df_country.rename(columns={"Country/Region": "Country_Region"}, inplace=True)

# =========================
# 📌 Merge step
# =========================
df_cluster = pd.merge(
    df_country,
    world_subset,
    on="Country_Region",
    how="inner"   # inner = ناخد الدول المشتركة فقط (أدق للـ clustering)
)

# =========================
# 📌 Validation
# =========================
print("Cluster dataset shape:", df_cluster.shape)
print("Missing values:\n", df_cluster.isnull().sum())

Cluster dataset shape: (171, 20)
Missing values:
 Country_Region             0
Confirmed                  0
Deaths                     0
Recovered                  0
Active                     0
New_cases                  0
New_deaths                 0
New_recovered              0
Deaths___100_Cases         0
Recovered___100_Cases      0
Deaths___100_Recovered     0
Confirmed_last_week        0
1_week_change              0
1_week_%_increase          0
WHO_Region                 0
Population                 0
TotalCases                 0
TotalDeaths               12
TotalRecovered             3
Tests_1M_pop              14
dtype: int64


In [11]:
# =========================
# 📌 1. COPY + BASIC FIXES
# =========================

df_full = df_full.copy()
df_clean = df_clean.copy()

# تحويل التاريخ
df_full['Date'] = pd.to_datetime(df_full['Date'])
df_clean['Date'] = pd.to_datetime(df_clean['Date'])

# =========================
# 📌 2. UNIFY COLUMN NAMES (VERY IMPORTANT)
# =========================

df_full.rename(columns={"Country/Region": "Country_Region"}, inplace=True)
df_clean.rename(columns={"Country/Region": "Country_Region"}, inplace=True)

# =========================
# 📌 3. AGGREGATE CLEAN DATA (Country Level)
# =========================

df_clean_agg = df_clean.groupby(
    ["Date", "Country_Region"]
)[["Confirmed", "Deaths", "Recovered", "Active"]].sum().reset_index()

# =========================
# 📌 4. EXTRACT GEO FEATURES (Lat / Long)
# =========================

clean_subset = df_clean[[
    "Country_Region", "Lat", "Long"
]].drop_duplicates()

# =========================
# 📌 5. MERGE (TIME SERIES BASE + GEO DATA)
# =========================

df_time = pd.merge(
    df_full,
    clean_subset,
    on="Country_Region",
    how="left"
)

# =========================
# 📌 6. VALIDATION
# =========================

print("Final shape:", df_time.shape)
print("\nMissing values:\n", df_time.isnull().sum())

df_time.head()

Final shape: (49068, 12)

Missing values:
 Date              0
Country_Region    0
Confirmed         0
Deaths            0
Recovered         0
Active            0
New_cases         0
New_deaths        0
New_recovered     0
WHO_Region        0
Lat               0
Long              0
dtype: int64


,Date,Country_Region,Confirmed,Deaths,Recovered,Active,New_cases,New_deaths,New_recovered,WHO_Region,Lat,Long
0,2020-01-22,Afghanistan,0,0,0,0,0,0,0,Eastern Mediterranean,33.93911,67.709953
1,2020-01-22,Albania,0,0,0,0,0,0,0,Europe,41.15330,20.168300
2,2020-01-22,Algeria,0,0,0,0,0,0,0,Africa,28.03390,1.659600
3,2020-01-22,Andorra,0,0,0,0,0,0,0,Europe,42.50630,1.521800
4,2020-01-22,Angola,0,0,0,0,0,0,0,Africa,-11.20270,17.873900


In [12]:
# =========================
# 📌 USA GEO-TIME DATASET
# =========================

df_usa["Date"] = pd.to_datetime(df_usa["Date"])

# ترتيب
df_usa = df_usa.sort_values(["Province_State", "Date"])

# =========================
# 📌 Validation
# =========================
print("USA dataset shape:", df_usa.shape)
print(df_usa.head())

/tmp/ipykernel_9882/4043247446.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_usa["Date"] = pd.to_datetime(df_usa["Date"])


USA dataset shape: (627920, 14)
         UID iso2 iso3  code3    FIPS   Admin2 Province_State Country_Region  \
82  84001001   US  USA    840  1001.0  Autauga        Alabama             US   
83  84001003   US  USA    840  1003.0  Baldwin        Alabama             US   
84  84001005   US  USA    840  1005.0  Barbour        Alabama             US   
85  84001007   US  USA    840  1007.0     Bibb        Alabama             US   
86  84001009   US  USA    840  1009.0   Blount        Alabama             US   

          Lat      Long_          Combined_Key       Date  Confirmed  Deaths  
82  32.539527 -86.644082  Autauga, Alabama, US 2020-01-22          0       0  
83  30.727750 -87.722071  Baldwin, Alabama, US 2020-01-22          0       0  
84  31.868263 -85.387129  Barbour, Alabama, US 2020-01-22          0       0  
85  32.996421 -87.125115     Bibb, Alabama, US 2020-01-22          0       0  
86  33.982109 -86.567906   Blount, Alabama, US 2020-01-22          0       0  


In [13]:
# =========================
# 📌 USA AGGREGATION (IMPORTANT)
# =========================

df_usa_agg = df_usa.groupby("Date")[["Confirmed", "Deaths"]].sum().reset_index()

df_usa_agg = df_usa_agg.sort_values("Date")

print(df_usa_agg.head())
print(df_usa_agg.shape)

        Date  Confirmed  Deaths
0 2020-01-22          1       0
1 2020-01-23          1       0
2 2020-01-24          2       0
3 2020-01-25          2       0
4 2020-01-26          5       0
(188, 3)


In [ ]:
# =========================
# 📌 FINAL CHECKS (VERY IMPORTANT)
# =========================

datasets = {
    "cluster": df_cluster,
    "time": df_time,
    "usa": df_usa,
    "day": df_day
}

for name, df in datasets.items():
    print("\n", "="*30)
    print(name)
    print("shape:", df.shape)
    print("missing %:\n", df.isnull().mean().sort_values(ascending=False).head(5))


cluster
shape: (171, 20)
missing %:
 Tests_1M_pop      0.081871
TotalDeaths       0.070175
TotalRecovered    0.017544
Country_Region    0.000000
Recovered         0.000000
dtype: float64

time
shape: (35156, 10)
missing %:
 Date              0.0
Country_Region    0.0
Confirmed         0.0
Deaths            0.0
Recovered         0.0
dtype: float64

usa
shape: (627920, 14)
missing %:
 FIPS      0.002994
Admin2    0.001796
iso2      0.000000
UID       0.000000
code3     0.000000
dtype: float64

day
shape: (188, 12)
missing %:
 Date         0.0
Confirmed    0.0
Deaths       0.0
Recovered    0.0
Active       0.0
dtype: float64


Clustering pipeline ready

In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# =========================
# 📌 CLUSTERING PIPELINE
# Datasets used:
# - country_wise_latest
# - worldometer_data
# هدفه: clustering countries based on COVID severity
# =========================

def clustering_pipeline(df_country, df_world):

    # =========================
    # 1) PREPROCESSING
    # =========================

    df_country = df_country.copy()
    df_world = df_world.copy()

    # unify column names
    df_country.rename(columns={"Country/Region": "Country_Region"}, inplace=True)

    # remove duplicates
    df_country = df_country.drop_duplicates()
    df_world = df_world.drop_duplicates()

    # merge datasets
    df = pd.merge(df_country, df_world, on="Country_Region", how="inner")

    # handle missing values
    df = df.fillna(df.median(numeric_only=True))

    # =========================
    # 2) FEATURE ENGINEERING
    # =========================

    df["death_rate"] = df["Deaths"] / (df["Confirmed"] + 1)
    df["recovery_rate"] = df["Recovered"] / (df["Confirmed"] + 1)
    df["cases_per_million"] = df["TotalCases"] / (df["Population"] + 1)

    # =========================
    # 3) SCALING
    # =========================

    features = df.select_dtypes(include=np.number)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(features)

    print("Clustering pipeline ready:", X_scaled.shape)

    return X_scaled, df

PIPELINE — TIME SERIES (full_grouped


In [16]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# =========================
# 📌 TIME SERIES PIPELINE (GLOBAL)
# Datasets:
# - full_grouped
# - covid_19_clean_complete
# هدفه: forecasting COVID trends over time
# =========================

def time_series_pipeline(df_full, df_clean):

    # =========================
    # 1) PREPROCESSING
    # =========================

    df_full = df_full.copy()
    df_clean = df_clean.copy()

    df_full["Date"] = pd.to_datetime(df_full["Date"])
    df_clean["Date"] = pd.to_datetime(df_clean["Date"])

    # aggregate clean dataset (country level → global)
    df_clean = df_clean.groupby("Date")[["Confirmed","Deaths","Recovered","Active"]].sum().reset_index()

    # unify datasets
    df = pd.concat([df_full, df_clean]).groupby("Date").sum().reset_index()

    df = df.sort_values("Date")

    # =========================
    # 2) FEATURE ENGINEERING
    # =========================

    df["lag_1"] = df["Confirmed"].shift(1)
    df["lag_7"] = df["Confirmed"].shift(7)
    df["rolling_mean_7"] = df["Confirmed"].rolling(7).mean()
    df["growth_rate"] = df["Confirmed"].pct_change()

    df = df.dropna()

    # =========================
    # 3) SCALING
    # =========================

    features = df.drop(["Date"], axis=1)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(features)

    print("Time Series pipeline ready:", X_scaled.shape)

    return X_scaled, df

PIPELINE — USA DATA

In [19]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# =========================
# 📌 USA TIME SERIES PIPELINE
# Dataset:
# - usa_country_wise (county-level data)
# هدفه: US-level forecasting
# =========================

def usa_pipeline(df_usa):

    # =========================
    # 1) PREPROCESSING
    # =========================

    df = df_usa.copy()

    df["Date"] = pd.to_datetime(df["Date"])

    # aggregate to national level
    df = df.groupby("Date")[["Confirmed","Deaths"]].sum().reset_index()

    df = df.sort_values("Date")

    # =========================
    # 2) FEATURE ENGINEERING
    # =========================

    df["lag_1"] = df["Confirmed"].shift(1)
    df["lag_7"] = df["Confirmed"].shift(7)
    df["rolling_mean_7"] = df["Confirmed"].rolling(7).mean()
    df["trend"] = df["Confirmed"].diff()

    df = df.dropna()

    # =========================
    # 3) SCALING
    # =========================

    features = df.drop(["Date"], axis=1)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(features)

    print("USA pipeline ready:", X_scaled.shape)

    return X_scaled, df

In [20]:
import pandas as pd

# =========================
# 📌 GLOBAL TRENDS PIPELINE
# Dataset: day_wise
# Purpose: Visualization & insights (NO ML)
# =========================

df_trends = df_day.copy()

# Convert date
df_trends["Date"] = pd.to_datetime(df_trends["Date"])

# Sort values
df_trends = df_trends.sort_values("Date")

print("Shape:", df_trends.shape)
print(df_trends.head())

Shape: (188, 12)
        Date  Confirmed  Deaths  Recovered  Active  New_cases  New_deaths  \
0 2020-01-22        555      17         28     510          0           0   
1 2020-01-23        654      18         30     606         99           1   
2 2020-01-24        941      26         36     879        287           8   
3 2020-01-25       1434      42         39    1353        493          16   
4 2020-01-26       2118      56         52    2010        684          14   

   New_recovered  Deaths___100_Cases  Recovered___100_Cases  \
0              0                3.06                   5.05   
1              2                2.75                   4.59   
2              6                2.76                   3.83   
3              3                2.93                   2.72   
4             13                2.64                   2.46   

   Deaths___100_Recovered  No._of_countries  
0                   60.71                 6  
1                   60.00                 8  
2  

🔵 System 1: Country Clustering
   → country_wise_latest + worldometer_data

🔴 System 2: Time Series Forecasting
   → full_grouped (+ clean_complete optional)

🟡 System 3: USA Geo Analysis
   → usa_country_wise

⚪ System 4: Global Trends
   → day_wise (visual only)